# GEOM90006
# Assignment 3 - Formative Task 
## Raster Data

This is a assignment to be completed in Jupyter Lab. It will be autograded. A library `nbgrager` has been used for this autograding. 

<div style="background-color:#E0EDFA;padding:1em">
You should complete this task in Jupyter Notebook itself, not a third-party editor.
</div>

There are a few major things that you need to know about autograding using `nbgrader`: 
1. You need to write your answers in the specified cells: cells where you see <b>"YOUR CODE HERE"</b>. - These cells are called `autograded answer.`
2. There are test cases which will validate your answers with multiple test case scenarios. - These cells are called `autograded tests.`
3. You will earn designated point with successful validation of each test cell.

<div style="background-color:#E0EDFA;padding:1em">
You will submit the completed assignment in LMS, as a jupyter notebook (.ipynb file). 

Rename your file as: <b><i>studentid_A3Formative.ipynb</i></b>

e.g., 101010_A3Formative.ipynb
</div>

## Basic Introduction to Raster data

<b>Raster data</b> consists of pixels, also known as grid cells, which are typically evenly spaced and square, although this is not mandatory. Rasters often appear pixelated due to each pixel having its own value or class.

In a satellite image, each pixel value comprises red, green, and blue components, while in an elevation map, each value denotes a specific height, ranging from rainfall to land cover.

Raster models are advantageous for storing continuously varying data, such as elevation surfaces, temperature, and lead contamination.

In this formative task, you will perform basic manipulation tasks on a continuous, single-band raster.

## Question F1 - Loading Raster File

Write a function named `get_raster_metadata` that utilizes the <b>rasterio</b> library to load the raster file  "`Victoria rainfall grid.asc`". 

The function should determine and return the following attributes of the raster:
- Coordinate Reference System (CRS)
- Resolution
- Height
- Width
- Number of bands

The function must return these values based on the variable text specified at the time of the function call.

<b>Use filepath "data/Victoria rainfall grid.asc" for autograder to work correctly.</b> 

In [10]:
import numpy as np

# package to read and write geospatial raster data
import rasterio
import rasterio.crs

def get_raster_metadata(raster_path, var):
    """
    Loads a raster file and returns specified metadata or property.

    Parameters:
    - raster_path (str): The file path to the raster dataset.
    - var (str): A string specifying which piece of raster metadata or property to return. 
                 Options are 'crs', 'res', 'height', 'width', 'count'.

    Returns:
    - The requested metadata or property of the raster dataset. 
      This can be the CRS, resolution, height, width, or band count of the raster.
    """
    # YOUR CODE HERE
    with rasterio.open(raster_path) as src:
        if var == 'crs':
            return src.crs
        elif var == 'res':
            return src.res
        elif var == 'height':
            return src.height
        elif var == 'width':
            return src.width
        elif var == 'count':
            return src.count
        else:
            raise NotImplementedError()

    

In [12]:
"""Check that get_raster_metadata returns the correct output for several inputs"""

raster_path = "data/Victoria rainfall grid.asc"

#Test case 1 - test crs
assert get_raster_metadata(raster_path, 'crs') ==  4283

#Test case 2- resolution
assert get_raster_metadata(raster_path, 'res') == (0.05,0.05)

#Test case 3 - height
assert get_raster_metadata(raster_path, 'height') == 104

#Test case 4 - width
assert get_raster_metadata(raster_path, 'width') == 181

#Test case 5 - band count
assert get_raster_metadata(raster_path,'count') == 1

## Question F2 - Calculating Raster Band Statistics



Write a function `band_sum` using the rasterio and numpy libraries. This function should:
1. Open a raster file "Victoria rainfall grid.asc" (<i> path has been provided for consistency</i>)
2. Read the <b>first band</b> of the raster dataset.
3. Replace all cells that contain the NoData value (-999) with NaN (Not a Number) to exclude them from statistical calculations.
4. Calculate and return the sum of all <b>non-NaN</b> values in the first band.

   <b>Note: There is a single test case for this question which is hidden. Please ensure you follow all four steps precisely to obtain the correct result. </b> 

In [16]:
import os
import rasterio
import numpy as np

def band_sum(raster_path):
    """
    Calculates the sum of the values in the first band of a raster file, ignoring any cells with a NoData value.

    Parameters:
    - raster_path (str): The file path to the raster dataset.

    Returns:
    - float: The sum of the values in the first band, excluding NoData values.
    """
    
    # YOUR CODE HERE
    with rasterio.open(raster_path) as src:
        band = src.read(1)
        
        no_data_value = src.nodata
        
        band = np.where(band == no_data_value, np.nan, band)
        
        return np.nansum(band)

In [18]:
# Raster path provided for test case
raster_path = 'data/Victoria rainfall grid.asc'

## Question F3 - Extract value from raster

Write a function named `extract_rainfall` that extracts the rainfall value from the raster file "Victoria rainfall grid.asc" at a given <b>latitude and longitude</b>. 

This function should perform the following steps:
1. <b>Locate</b> the corresponding row and column in the raster for the provided georeferenced coordinates.
2. <b>Check</b> whether the specified coordinates fall within the bounds of the raster.
3. If the coordinates are within bounds, <b>return the rainfall value</b> from the raster at that location.
4. If the coordinates are <b>outside the raster bounds</b>, return None.

   <b> Note: There are positive and negative test cases for this question, which are hidden.
   -  "Positive Test Case: Valid coordinates - should return the corresponding rainfall value."
   -  "Negative Test Case: Invalid coordinates (outside raster) - should return  is None."

In [26]:
import rasterio
from rasterio.transform import from_origin
from rasterio.enums import Resampling
import pyproj


def extract_rainfall(raster_path, lat, lon):
    """
    Extracts the rainfall value from a raster at a specific latitude and longitude.

    Parameters:
    - raster_path (str): Path to the raster file.
    - lat (float): Latitude of the point of interest.
    - lon (float): Longitude of the point of interest.

    Returns:
    - float or None: Rainfall value at the specified location, or None if the point is outside the raster bounds.
    """

    # YOUR CODE HERE
    with rasterio.open(raster_path) as src:
        row, col = src.index(longitude, latitude)
        if 0 <= row < src.height and 0 <= col < src.width:
            band_data = src.read(1)
            
            value = band_data[row, col]

            no_data_value = src.nodata
            if value == no_data_value:
                return None
            else:   
                return float(value)
        else:
            return None

In [28]:
# Raster path and coordinates provided for test cases
raster_path = 'data/Victoria rainfall grid.asc'

# Coordinates of interest (Melbourne, Victoria)
lat, lon = -37.81, 144.96


## Question F4 - Reprojecting raster

Write a function named `reproject_raster` that reprojects the raster file "Victoria rainfall grid.asc"  to a new <b> Coordinate Reference System (CRS)</b> using the `rasterio` library. The function must then return the value of the reprojected raster at a specific easting and northing location. 

The function should accept the following parameters:
- input_raster_path (str): Path to the input raster
- output_raster_path (str): Path where the reprojected raster should be saved
- destination_crs (str or dict): The target CRS for reprojection (e.g., "EPSG:7855")
- easting (float): The easting coordinate (X) in the target CRS
- northing (float): The northing coordinate (Y) in the target CRS

Steps to be followed (hints):
1. Open the input raster file for reading.
2. Reproject the raster using bilinear resampling.
3. Write the reprojected raster to the specified output path.  
4. Read the newly created output raster.
5. Return the raster value at the specified easting and northing.

<b>Please note: There is a single hidden test case for this question. Be sure to follow all steps and use bilinear interpolation to pass the test successfully.</b> 

In [30]:
import os
import rasterio
import rasterio.warp
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject

def reproject_raster(input_raster_path, output_raster_path, destination_crs, easting, northing):

    """
    Reproject a raster file to a new coordinate reference system (CRS).

    Parameters:
    - input_raster_path: path to the input raster file. (input_raster_path has been provided below in autograded tests cell.)
    - output_raster_path: path where the reprojected raster will be saved. (output_raster_path has been provided below in autograded tests cell.)
    - destination_crs: the new CRS to reproject the raster to.  
    - easting : easting value is passed during the function call from the hidden test case.
    - northing : northing value is passed during the function call from the hidden test case.

    Returns:
    Value of the reprojected raster at the provided easting and northing. 
    """
    
    # YOUR CODE HERE
    with rasterio.open(input_raster_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, destination_crs, src.width, src.height, *src.bounds
        )
        
        kwargs = src.meta.copy()
        kwargs.update({'crs': destination_crs, 'transform': transform, 'width': width, 'height': height})
        with rasterio.open(output_raster_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=destination_crs,
                    resampling=Resampling.bilinear
                )
    
    with rasterio.open(output_raster_path) as dst:
        row, col = dst.index(easting, northing)
        return float(dst.read(1)[row, col])

In [32]:
#Raster paths and destination crs provided for test case

input_raster_path = 'data/Victoria rainfall grid.asc'
output_raster_file = './output/Victoria rainfall grid projected.tif'
destination_crs = 7855
